# CYK Parser Success Rate Evaluation

This notebook evaluates the CYK parser success rate across all California committee hearings in the corpus.

In [1]:
from pathlib import Path
import subprocess
import zipfile

# This is where the unzipped corpus file is stored
CORPUS_FILE_PATH = 'DH2024_Corpus_Release/'
corpus_dir = Path(CORPUS_FILE_PATH)
zip_path = Path("digitaldemocracy-2015-2018/DH2024_Corpus_Release.zip")
repo_dir = Path("digitaldemocracy-2015-2018")

# Clone repository if not present
if not repo_dir.is_dir():
    print("Corpus directory not found. Cloning repository...")
    subprocess.run(
        ["git", "clone", "https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018"],
        check=True,
    )
    print("Repository cloned successfully.")

# Extract zip file if corpus directory doesn't exist
if not corpus_dir.is_dir():
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete.")
    else:
        print(f"Error: {zip_path} not found!")
else:
    print(f"Corpus already extracted at {corpus_dir}")

Corpus already extracted at DH2024_Corpus_Release


## Load California Hearings

In [2]:
from src import HearingLoader, HearingTagger
from src.grammar.Parser import Parser
import time

# Initialize the HearingLoader with the corpus path
loader = HearingLoader(corpus_path='DH2024_Corpus_Release/')
tagger = HearingTagger()
parser = Parser()

print("Loading all committee hearings...")
start_time = time.time()
all_hearings = loader.load_all_committee_hearings()
load_time = time.time() - start_time
print(f"Total hearings loaded: {len(all_hearings)} (took {load_time:.1f}s)")

# Filter for California hearings only
ca_hearings = [h for h in all_hearings if h.state == 'CA']
print(f"California hearings: {len(ca_hearings)}")

# Filter out hearings with no bill discussed
ca_hearings_with_bills = [h for h in ca_hearings if h.bid != 'CA_NO BILL DISCUSSED'][:500]
print(f"California hearings with bills: {len(ca_hearings_with_bills)}")

Loading all committee hearings...
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
Total hearings loaded: 8218 (took 10.6s)
California hearings: 8218
California hearings with bills: 500


## Run Tagger and Evaluate CYK Parser (Combined)

To improve efficiency, we'll tag and parse in a single loop with progress tracking.

In [ ]:
import numpy as np

print("Processing hearings (tagging + parsing)...")
print("This will take several minutes...\n")

parse_results = []
successful_parses = []
failed_parses = []
tagging_errors = []

total = len(ca_hearings_with_bills)
start_time = time.time()

for i, hearing in enumerate(ca_hearings_with_bills):
    # Progress update every 50 hearings
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (total - i - 1) / rate if rate > 0 else 0
        print(f"Processed {i + 1}/{total} hearings ({(i+1)/total*100:.1f}%)")
    
    try:
        # Tag the hearing
        tagged_hearing = tagger(hearing)
        
        # Try to parse
        try:
            # Build token sequence from hearing utterances
            token_seq = [
                tagged_hearing.speakers[utterance.pid].speaker_position
                for utterance in tagged_hearing.utterances
            ]
            
            parse_trees = list(parser.get_all_parses_as_nltk_trees(token_seq, max_parses=1) or [])
            parse_success = len(parse_trees) > 0
            
            parse_results.append(int(parse_success))
            
            if parse_success:
                successful_parses.append((hearing.hid, hearing.bid, len(parse_trees)))
            else:
                failed_parses.append((hearing.hid, hearing.bid))
                
        except Exception as e:
            # Parsing failed with exception
            parse_results.append(0)
            failed_parses.append((hearing.hid, hearing.bid))
            
    except Exception as e:
        # Tagging failed
        tagging_errors.append((hearing.hid, hearing.bid, str(e)))
        continue

total_time = time.time() - start_time
print(f"\nProcessing complete! Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Successfully processed: {len(parse_results)}/{total} hearings")
print(f"Tagging errors: {len(tagging_errors)}")

In [4]:
tagging_errors

[(52363, 'CA_201720180AB1246', 'min() iterable argument is empty'),
 (254780, 'CA_201720180AB2970', '100929'),
 (255466, 'CA_201720180SB910', 'min() iterable argument is empty')]

## Results

In [5]:
total_hearings = len(parse_results)
successful_count = np.sum(parse_results)
failed_count = total_hearings - successful_count
success_rate = np.mean(parse_results) * 100 if parse_results else 0

print("=" * 80)
print("CYK Parser Statistics for 2015-2018 California Hearings")
print("=" * 80)
print(f"\nTotal hearings evaluated: {total_hearings}")
print(f"Successful parses:\t{successful_count}")
print(f"Failed parses:\t{failed_count}")
print(f"\nSuccess rate:\t{success_rate:.2f}%")
print("\n" + "=" * 80)

CYK Parser Statistics for 2015-2018 California Hearings

Total hearings evaluated: 497
Successful parses:	400
Failed parses:	97

Success rate:	80.48%



## Detailed Statistics

In [6]:
successful_parses[:5]

[(52054, 'CA_201720180AB816', 1),
 (52054, 'CA_201720180AB822', 1),
 (52054, 'CA_201720180AB677', 1),
 (52054, 'CA_201720180AB12', 1),
 (52363, 'CA_201720180AB262', 1)]

In [7]:
print("\nSuccessful Parses Statistics:")
print(f"  Total: {len(successful_parses)}")

if successful_parses:
    parse_tree_counts = [count for _, _, count in successful_parses]
    print(f"  Average parse trees per hearing: {np.mean(parse_tree_counts):.2f}")
    print(f"  Min parse trees: {np.min(parse_tree_counts)}")
    print(f"  Max parse trees: {np.max(parse_tree_counts)}")
    
    # Show distribution of parse tree counts
    unique, counts = np.unique(parse_tree_counts, return_counts=True)
    print("\n  Parse tree count distribution:")
    for tree_count, freq in zip(unique, counts):
        print(f"    {tree_count} tree(s): {freq} hearings ({freq/len(successful_parses)*100:.1f}%)")

print(f"\nFailed Parses:")
print(f"  Total: {len(failed_parses)}")

# Show first 10 failed parses as examples
if failed_parses:
    print("\n  Examples of failed parses (first 10):")
    for i, (hid, bid) in enumerate(failed_parses[:10]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")

if tagging_errors:
    print(f"\nTagging Errors:")
    print(f"  Total: {len(tagging_errors)}")
    print("\n  Examples (first 5):")
    for i, (hid, bid, error) in enumerate(tagging_errors[:5]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")
        print(f"       Error: {error[:100]}..." if len(error) > 100 else f"       Error: {error}")


Successful Parses Statistics:
  Total: 400
  Average parse trees per hearing: 1.00
  Min parse trees: 1
  Max parse trees: 1

  Parse tree count distribution:
    1 tree(s): 400 hearings (100.0%)

Failed Parses:
  Total: 97

  Examples of failed parses (first 10):
    1. Hearing ID: 52054, Bill: CA_201720180AB92
    2. Hearing ID: 52054, Bill: CA_201720180AB547
    3. Hearing ID: 52363, Bill: CA_201720180AB1223
    4. Hearing ID: 52363, Bill: CA_201720180AB77
    5. Hearing ID: 52780, Bill: CA_201720180AB262
    6. Hearing ID: 52773, Bill: CA_201720180AB749
    7. Hearing ID: 52773, Bill: CA_201720180AB32
    8. Hearing ID: 52773, Bill: CA_201720180AB632
    9. Hearing ID: 54113, Bill: CA_201720180SB574
    10. Hearing ID: 54013, Bill: CA_201720180SJR6

Tagging Errors:
  Total: 3

  Examples (first 5):
    1. Hearing ID: 52363, Bill: CA_201720180AB1246
       Error: min() iterable argument is empty
    2. Hearing ID: 254780, Bill: CA_201720180AB2970
       Error: 100929
    3. Hearing

# Failure Cases

In [8]:
# Load failed hearing examples for analysis
import random

random.seed(42)

# Sample 5 failed hearings for detailed analysis
sampled_failed_hids = random.sample(failed_parses, min(5, len(failed_parses)))
print(f"Sampled {len(sampled_failed_hids)} failed hearings for analysis:")
for hid, bid in sampled_failed_hids:
    print(f"  - Hearing ID: {hid}, Bill: {bid}")

# Load the full hearings
failed_hearing_objects = []
for hid, bid in sampled_failed_hids:
    # Find the hearing in our loaded data
    for h in ca_hearings_with_bills:
        if h.hid == hid and h.bid == bid:
            # Tag it
            try:
                tagged = tagger(h)
                if tagged:
                    failed_hearing_objects.append(tagged)
                    break
            except Exception as e:
                print(f"Error tagging {hid}/{bid}: {e}")
                break

print(f"\nSuccessfully loaded {len(failed_hearing_objects)} failed hearings")

Sampled 5 failed hearings for analysis:
  - Hearing ID: 255513, Bill: CA_201720180SB1076
  - Hearing ID: 52997, Bill: CA_201720180AB614
  - Hearing ID: 52363, Bill: CA_201720180AB77
  - Hearing ID: 254615, Bill: CA_201720180AB2388
  - Hearing ID: 52296, Bill: CA_201720180AB225
Error tagging 255513/CA_201720180SB1076: min() iterable argument is empty
Error tagging 52296/CA_201720180AB225: min() iterable argument is empty

Successfully loaded 3 failed hearings


In [ ]:
# Export failed hearing token sequences
import csv
import nltk
from nltk import bigrams, trigrams
from collections import Counter

# Get all failed hearings token sequences
print("Collecting token sequences from all failed hearings...")
all_failed_token_sequences = []
failed_sequence_data = []

for hid, bid in failed_parses:
    for h in ca_hearings_with_bills:
        if h.hid == hid and h.bid == bid:
            try:
                tagged = tagger(h)
                if tagged:
                    # Build token sequence from hearing utterances
                    tokens = [
                        tagged.speakers[utterance.pid].speaker_position
                        for utterance in tagged.utterances
                    ]
                    token_names = [token.name for token in tokens]
                    all_failed_token_sequences.append(token_names)
                    failed_sequence_data.append({
                        'hid': hid,
                        'bid': bid,
                        'token_count': len(tokens),
                        'token_sequence': ' -> '.join(token_names)
                    })
                    break
            except Exception as e:
                # Skip hearings that fail to tag
                break

print(f"Collected {len(all_failed_token_sequences)} failed hearing token sequences")

# Export to CSV
output_csv_file = "failed_parse_token_sequences.csv"
with open(output_csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['hid', 'bid', 'token_count', 'token_sequence'])
    writer.writeheader()
    writer.writerows(failed_sequence_data)

print(f"Token sequences exported to: {output_csv_file}")

# Analyze bigrams and trigrams
print("\n" + "=" * 80)
print("NLTK N-gram Analysis of Failed Parse Token Sequences")
print("=" * 80)

# Flatten all token sequences for n-gram analysis
all_tokens_flat = [token for seq in all_failed_token_sequences for token in seq]
print(f"\nTotal tokens in failed hearings: {len(all_tokens_flat)}")

# Bigram analysis
print("\n" + "-" * 80)
print("Most Common Bigrams (token pairs):")
print("-" * 80)
bigram_list = list(bigrams(all_tokens_flat))
bigram_freq = Counter(bigram_list)

for i, (bigram, count) in enumerate(bigram_freq.most_common(20), 1):
    percentage = (count / len(bigram_list)) * 100
    print(f"{i:2d}. {bigram[0]:20s} -> {bigram[1]:20s}  ({count:3d} occurrences, {percentage:5.2f}%)")

# Trigram analysis
print("\n" + "-" * 80)
print("Most Common Trigrams (token triples):")
print("-" * 80)
trigram_list = list(trigrams(all_tokens_flat))
trigram_freq = Counter(trigram_list)

for i, (trigram, count) in enumerate(trigram_freq.most_common(20), 1):
    percentage = (count / len(trigram_list)) * 100
    print(f"{i:2d}. {trigram[0]:20s} -> {trigram[1]:20s} -> {trigram[2]:20s}  ({count:3d} occurrences, {percentage:5.2f}%)")

print("\n" + "=" * 80)

## Compare Successful vs Failed Hearings

Let's compare the characteristics of successful vs failed parses to identify what makes a hearing unparseable.

In [ ]:
from collections import Counter

# Analyze all failed hearings (not just the sample)
print("Analyzing all failed hearing patterns...")
print("=" * 80)

# Get all failed hearings
all_failed_hearings = []
for hid, bid in failed_parses:
    for h in ca_hearings_with_bills:
        if h.hid == hid and h.bid == bid:
            try:
                tagged = tagger(h)
                if tagged:
                    all_failed_hearings.append(tagged)
                    break
            except Exception as e:
                # Skip hearings that fail to tag
                break

print(f"Successfully loaded {len(all_failed_hearings)} failed hearings for pattern analysis\n")

# Analyze token sequence lengths
failed_token_lengths = []
failed_token_sequences = []
failed_section_counts = []

for hearing in all_failed_hearings:
    # Build token sequence from hearing utterances
    tokens = [
        hearing.speakers[utterance.pid].speaker_position
        for utterance in hearing.utterances
    ]
    token_names = tuple([token.name for token in tokens])
    
    failed_token_lengths.append(len(tokens))
    failed_token_sequences.append(token_names)

# Token length statistics
print("Token Length Statistics for Failed Hearings:")
print(f"  Mean: {np.mean(failed_token_lengths):.1f}")
print(f"  Median: {np.median(failed_token_lengths):.1f}")
print(f"  Min: {np.min(failed_token_lengths)}")
print(f"  Max: {np.max(failed_token_lengths)}")

# Most common token sequences in failures
print("\nMost Common Token Sequences in Failed Hearings (top 10):")
sequence_counter = Counter(failed_token_sequences)
for i, (seq, count) in enumerate(sequence_counter.most_common(10), 1):
    print(f"  {i}. [{' -> '.join(seq)}] ({count} occurrences)")

# Most common starting tokens
starting_tokens = [seq[0] if seq else None for seq in failed_token_sequences]
print("\nMost Common Starting Tokens in Failed Hearings:")
starting_counter = Counter(starting_tokens)
for token, count in starting_counter.most_common():
    if token:
        print(f"  {token}: {count} ({count/len(starting_tokens)*100:.1f}%)")

# Most common ending tokens
ending_tokens = [seq[-1] if seq else None for seq in failed_token_sequences]
print("\nMost Common Ending Tokens in Failed Hearings:")
ending_counter = Counter(ending_tokens)
for token, count in ending_counter.most_common():
    if token:
        print(f"  {token}: {count} ({count/len(ending_tokens)*100:.1f}%)")

## Visualize Section Speaker Rule Requirements

Compare failed hearings against the grammar's section speaker position requirements.

In [ ]:
# Analyze token sequences of failed hearings
print("Failed Hearing Token Sequences")
print("=" * 80)

for i, hearing in enumerate(failed_hearing_objects):
    print(f"\n{'='*80}")
    print(f"Failed Hearing {i+1}")
    print(f"{'='*80}")
    print(f"Bill: {hearing.bid}")
    print(f"Hearing ID: {hearing.hid}")
    
    # Build token sequence from hearing utterances
    tokens = [
        hearing.speakers[utterance.pid].speaker_position
        for utterance in hearing.utterances
    ]
    
    # Show token sequence
    print(f"\nToken count: {len(tokens)}")
    print("\nToken sequence (SpeakerPositionEnum):")
    token_names = [token.name for token in tokens]
    print("  " + " -> ".join(token_names))